# Chapter 4: Short chains and the coefficient 37/768

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## Why the first correction is of order $\lambda^2$

For a chain much shorter than $p$, a tangent that starts at an angle bounded away from $\pi/2$ keeps its
sign along $\hat{\mathbf e}$ over the whole contour, so the projected contour is monotone and its span
equals its projected path length. The deficit $\lambda/2-F$ comes from **grazing** orientations,
$U_0=O(\sqrt\lambda)$. Writing $U_{\lambda t}=\sqrt\lambda\,\xi_t$ for $0\le t\le1$, the tangent SDE becomes
$d\xi=dW+O(\lambda)$, and $Z_{\lambda t}-Z_0=\lambda^{3/2}\int_0^t\xi_s\,ds$: the tangent is Brownian and the
position is **integrated Brownian motion** (the randomly accelerated particle). The angular measure of
the grazing sector is $O(\sqrt\lambda)$, the range lost per grazing path is $O(\lambda^{3/2})$, and their
product is the $\lambda^2$ correction.

## The return kernel and the integral $J=37/192$

For velocity generator $\partial_v^2$, the first-return density of integrated Brownian motion from
outward speed $v_0$ to inward speed $v$ at time $t$ is (McKean 1963; Burkhardt 2014)
$$\mathcal K(v,v_0;t)=\frac{\sqrt3}{2\pi}\frac v{t^2}\exp\Big[-\frac{v^2-vv_0+v_0^2}t\Big]\operatorname{erf}\Big(\sqrt{\tfrac{3vv_0}t}\Big).$$
By scale invariance the speed-weighted return integral $J=\int_0^\infty v_0\,dv_0\int_0^\infty\mathcal K(v,v_0;1)\,dv$
does not depend on $t$. We evaluate it three ways.

In [2]:
from scipy import integrate
K = lambda v, v0, t=1.0: math.sqrt(3)/(2*math.pi)*v/t**2*math.exp(-(v*v-v*v0+v0*v0)/t)*math.erf(math.sqrt(3*v*v0/t))
J_num, err = integrate.dblquad(lambda v, v0: v0*K(v, v0), 0, 12, 0, 12)
print('direct 2D quadrature       J =', J_num, '+/-', err)
# substitution v = eta v0 and the a-integral in closed form gives a 1D integral
eta = sp.symbols('eta', positive=True)
J_eta = sp.Rational(9,8)/sp.pi*sp.integrate(eta**sp.Rational(3,2)*(eta**2+eta+1)/((eta**2-eta+1)**2*(eta+1)**3), (eta, 0, sp.oo))
print('symbolic eta-integral      J =', sp.nsimplify(sp.simplify(J_eta)))
r = sp.symbols('r', real=True)
J_r = sp.Rational(9,16)/sp.pi*sp.integrate((3+4*r**2)/((1+r**2)**2*(1+4*r**2)**2), (r, 0, sp.oo))
print('rational r-integral        J =', sp.simplify(J_r), '=', float(J_r))
print('37/192 =', 37/192)

direct 2D quadrature       J = 0.19270833332042725 +/- 1.4843992325327324e-08


symbolic eta-integral      J = 37/192


rational r-integral        J = 37/192 = 0.19270833333333334
37/192 = 0.19270833333333334


### The reduction of $J$ to a rational integral, step by step

1. **Scale invariance.** Under $v\to\sqrt t\,v$, $v_0\to\sqrt t\,v_0$ the kernel scales as
   $\mathcal K(v,v_0;t)=t^{-3/2}\mathcal K(v/\sqrt t,v_0/\sqrt t;1)$, and the measure $v_0\,dv_0\,dv$ as $t^{3/2}$,
   so $J$ is the same at every $t$. The speed-weighted probability of a return within unit time is
   therefore $\int_0^1J\,dt=J$.
2. **Polar-type substitution.** Put $v=\eta v_0$ at $t=1$:
   $J=\frac{\sqrt3}{2\pi}\int_0^\infty\eta\,d\eta\int_0^\infty v_0^3\,e^{-\alpha v_0^2}\operatorname{erf}(\beta v_0)\,dv_0$,
   with $\alpha=\eta^2-\eta+1$ and $\beta=\sqrt{3\eta}$.
3. **The inner integral** is elementary:
   $\int_0^\infty a^3e^{-\alpha a^2}\operatorname{erf}(\beta a)\,da=\dfrac{\beta}{2\alpha^2\sqrt{\alpha+\beta^2}}+\dfrac{\beta}{4\alpha(\alpha+\beta^2)^{3/2}}$
   (minus the derivative with respect to $\alpha$ of $\int_0^\infty a\,e^{-\alpha a^2}\operatorname{erf}(\beta a)\,da=\beta/[2\alpha\sqrt{\alpha+\beta^2}]$).
4. **The square.** $\alpha+\beta^2=\eta^2+2\eta+1=(\eta+1)^2$, so the square roots disappear and the bracket
   becomes $\sqrt{3\eta}\,\dfrac{2(\eta+1)^2+\alpha}{4\alpha^2(\eta+1)^3}=\sqrt{3\eta}\,\dfrac{3(\eta^2+\eta+1)}{4\alpha^2(\eta+1)^3}$,
   which gives the first form of $J$ quoted in the article, with the prefactor $\frac{\sqrt3}{2\pi}\cdot\frac{3\sqrt3}{4}=\frac9{8\pi}$.
5. **Rational form.** With $r=(\sqrt\eta-1/\sqrt\eta)/2$, i.e. $\sqrt\eta=r+\sqrt{1+r^2}$, the integrand is
   rational and even in $r$, and $0<\eta<\infty$ maps to $-\infty<r<\infty$; partial fractions give $37/192$.

The cell checks steps 1, 3, 4 and 5 separately.

In [3]:
import mpmath as mp
# step 1: J does not depend on t
for tt in (0.5, 1.0, 2.0):
    Jt = integrate.dblquad(lambda v, v0: v0*K(v, v0, tt), 0, 12*math.sqrt(tt), 0, 12*math.sqrt(tt))[0]
    print(f'step 1:  J(t={tt}) = {Jt:.10f}')
# step 3: the inner integral, at arbitrary alpha and beta
for al, be in ((1.7, 0.9), (0.75, 1.2247), (3.0, 2.4)):
    lhs = mp.quad(lambda a_: a_**3*mp.exp(-al*a_**2)*mp.erf(be*a_), [0, mp.inf])
    rhs = be/(2*al**2*mp.sqrt(al+be**2)) + be/(4*al*(al+be**2)**1.5)
    print(f'step 3:  alpha={al}, beta={be}:  quadrature {float(lhs):.12f}   closed form {float(rhs):.12f}')
# step 4: the algebra, symbolically
eta = sp.symbols('eta', positive=True)
al, be = eta**2 - eta + 1, sp.sqrt(3*eta)
print('step 4:  alpha + beta^2 - (eta+1)^2 =', sp.expand(al + be**2 - (eta+1)**2))
bracket = be/(2*al**2*(eta+1)) + be/(4*al*(eta+1)**3)
integrand = sp.sqrt(3)/(2*sp.pi)*eta*bracket
target = sp.Rational(9, 8)/sp.pi*eta**sp.Rational(3, 2)*(eta**2+eta+1)/(al**2*(eta+1)**3)
print('         integrand - first form of J =', sp.simplify(integrand - target))
# step 5: the substitution sqrt(eta) = r + sqrt(1+r^2), including the Jacobian
r = sp.symbols('r', real=True)
eta_r = (r + sp.sqrt(1+r**2))**2
lhs = target.subs(eta, eta_r)*sp.diff(eta_r, r)
rhs = sp.Rational(9, 16)/sp.pi*(3+4*r**2)/((1+r**2)**2*(1+4*r**2)**2)
ratios = [round(float((lhs/rhs).subs(r, x)), 12) for x in (-2, -0.3, 0, 0.7, 5)]
print('step 5:  [eta-integrand x d eta/dr] / [r-integrand] at r = -2, -0.3, 0, 0.7, 5:', ratios)
assert all(abs(x - 0.5) < 1e-10 for x in ratios)
print('         the ratio is 1/2 everywhere: eta from 0 to infinity maps to r from -infinity to infinity, and the r-integrand is even,')
print('         so the integral over the whole r-axis of half of it equals the integral from 0 to infinity in the second form of J')
print('         partial fractions:', sp.apart((3+4*r**2)/((1+r**2)**2*(1+4*r**2)**2), r))

step 1:  J(t=0.5) = 0.1927083333
step 1:  J(t=1.0) = 0.1927083333
step 1:  J(t=2.0) = 0.1927083333
step 3:  alpha=1.7, beta=0.9:  quadrature 0.131565907647   closed form 0.131565907647
step 3:  alpha=0.75, beta=1.2247:  quadrature 0.846732763023   closed form 0.846732763023
step 3:  alpha=3.0, beta=2.4:  quadrature 0.052763055574   closed form 0.052763055574
step 4:  alpha + beta^2 - (eta+1)^2 = 0
         integrand - first form of J = 0
step 5:  [eta-integrand x d eta/dr] / [r-integrand] at r = -2, -0.3, 0, 0.7, 5: [0.5, 0.5, 0.5, 0.5, 0.5]
         the ratio is 1/2 everywhere: eta from 0 to infinity maps to r from -infinity to infinity, and the r-integrand is even,
         so the integral over the whole r-axis of half of it equals the integral from 0 to infinity in the second form of J
         partial fractions: -16/(27*(4*r**2 + 1)) + 32/(9*(4*r**2 + 1)**2) + 4/(27*(r**2 + 1)) - 1/(9*(r**2 + 1)**2)


## From $J$ to the WLC coefficient

The WLC grazing layer has velocity generator $\tfrac12\partial_\xi^2$, not $\partial_v^2$; rescaling speeds by
$1/\sqrt2$ maps one to the other and multiplies the speed-weighted return by $\tfrac12$:
$\int_0^\infty\xi\,Q_{\rm ret}(\xi)\,d\xi=J/2=37/384$. Inserting the scaling form
$\int_0^1v\,q(0^+,-v;\lambda)\,dv=\lambda\int_0^\infty\xi Q_{\rm ret}(\xi)\,d\xi+o(\lambda)$ into the flux identity of
Chapter 2, $F'=\tfrac12-\tfrac{37}{384}\lambda+o(\lambda)$, and integrating,
$$F(\lambda)=\frac\lambda2-\frac{37}{768}\lambda^2+o(\lambda^2).$$
**Status: derived.** The scaling limit is taken under the integral by dominated convergence: a return
from outward speed $v$ requires the tangent to reach zero, and the exponential martingale inequality
gives $\Pr(T_v\le t)\le\exp[-(v-t)^2/(2t)]$ for $v>t$, hence the $t$-independent integrable envelope
$\xi Q_t(\xi)\le\xi\mathbf 1_{\xi\le2}+\xi e^{-\xi^2/8}\mathbf 1_{\xi>2}$. No rate for the remainder follows.

## Numerical evidence (Fig. 3a)

The refined spectral solutions at angular orders up to 512 with first-alias-corrected inversion
(`data/stiff_alias_summary.csv`) give the deficit $(\lambda/2-F)/\lambda^2$:

In [4]:
stiff = pd.read_csv(ROOT/'data'/'stiff_alias_summary.csv')
stiff['deficit'] = (stiff['lambda']/2 - stiff.F_extrap)/stiff['lambda']**2
print(stiff[['lambda', 'F_extrap', 'extrapolation_sensitivity', 'deficit']].to_string(index=False))
slope, intercept = np.polyfit(stiff['lambda'], stiff['deficit'], 1)
print(f'\nlinear fit: intercept {intercept:.6f} (37/768 = {37/768:.6f}),  slope {slope:.5f})')
check('intercept vs 37/768', intercept, 37/768, 2e-5)

 lambda   F_extrap  extrapolation_sensitivity    deficit
 0.0125 0.00624248             9.94280786e-12 0.04810784
 0.0250 0.01246998             1.93635212e-12 0.04803877
 0.0500 0.02488025             1.08249867e-12 0.04790109
 0.0750 0.03723133             2.76637047e-12 0.04776402
 0.1000 0.04952372             4.20451868e-12 0.04762756
 0.2000 0.09811649             8.39424363e-12 0.04708779

linear fit: intercept 0.048174 (37/768 = 0.048177),  slope -0.00544)
PASS  intercept vs 37/768: 0.04817373349  (expected 0.04817708333, tol 2e-05)


np.True_

### The limit function $Q_{\rm ret}$ and the dominated convergence step (supplementary material)

Equation (10) needs the limit $\lambda\to0$ under the integral over starting speeds. Two ingredients:

* **The limit function.** For the WLC layer (generator $\tfrac12\partial_\xi^2$), $Q_{\rm ret}(\xi)$ is the
  probability of a return within unit time from outward speed $\xi$. In McKean's units (generator
  $\partial_v^2$) the speed is $v_0=\sqrt2\,\xi$, and by the scaling of step 1,
  $Q_{\rm ret}(\xi)=\int_0^1dt\int_0^\infty\mathcal K(v,\sqrt2\xi;t)\,dv=2\int_{\sqrt2\xi}^\infty\frac{G(a)}a\,da$,
  with $G(a)=\int_0^\infty\mathcal K(w,a;1)\,dw$. Since $\xi\,d\xi=\tfrac12v_0\,dv_0$, this gives
  $\int_0^\infty\xi\,Q_{\rm ret}\,d\xi=J/2=37/384$.
* **The envelope.** From the tangent equation, $U_s=v-\int_0^sU_r\,dr+N_s\ge v-s+N_s$, where $N$ is a martingale with
  quadratic variation at most $s$. A return within $t$ requires $U$ to reach zero, hence $N_s\le-(v-t)$
  for some $s\le t$, and the exponential martingale inequality gives $\Pr(T_v\le t)\le e^{-(v-t)^2/(2t)}$
  for $v>t$. With $v=\sqrt t\,\xi$ and $t\le1$: $(v-t)^2/(2t)=(\xi-\sqrt t)^2/2\ge\xi^2/8$ for $\xi>2$. So
  $\xi\,Q_t(\xi)\le\xi\mathbf 1_{\{\xi\le2\}}+\xi e^{-\xi^2/8}\mathbf 1_{\{\xi>2\}}$ for every $t\le1$, which is integrable.

The cell evaluates $Q_{\rm ret}$, its speed-weighted integral, and the envelope. As a consistency check,
$Q_{\rm ret}(0^+)$ computed over *all* times must be 1 (a randomly accelerated particle returns with certainty);
in the present variables that is $2\int_0^\infty G(a)\,da/a=1$.

In [5]:
Kw = lambda w, a_: math.sqrt(3)/(2*math.pi)*w*math.exp(-(w*w-w*a_+a_*a_))*math.erf(math.sqrt(3*w*a_))
G = lambda a_: integrate.quad(lambda w: Kw(w, a_), 0, np.inf)[0]
Qmck = lambda v0: 2*integrate.quad(lambda a_: G(a_)/a_, v0, np.inf, limit=200)[0]      # return within unit time, McKean units
total = 4*integrate.quad(lambda x_: G(x_*x_)/x_, 0, np.inf, limit=200)[0]                  # a = x^2 removes the square-root singularity at a = 0
check('total return probability, 2 int_0^inf G(a)/a da', total, 1.0, 1e-6)
print('\n   xi     Q_ret(xi)      envelope')
for xi in (0.25, 0.5, 1, 2, 3, 4):
    print(f'{xi:5g}   {Qmck(math.sqrt(2)*xi):.4e}     {1.0 if xi <= 2 else math.exp(-xi*xi/8):.4e}')
val = integrate.quad(lambda xi: xi*Qmck(math.sqrt(2)*xi), 0, 8, limit=100)[0]
check('int xi Q_ret d xi = 37/384', val, 37/384, 1e-8)

PASS  total return probability, 2 int_0^inf G(a)/a da: 0.9999999999  (expected 1, tol 1e-06)

   xi     Q_ret(xi)      envelope
 0.25   4.2685e-01     1.0000e+00
  0.5   2.2118e-01     1.0000e+00
    1   4.3344e-02     1.0000e+00
    2   2.6668e-04     1.0000e+00
    3   1.0174e-07     3.2465e-01
    4   2.1331e-12     1.3534e-01


PASS  int xi Q_ret d xi = 37/384: 0.09635416665  (expected 0.09635416667, tol 1e-08)


True

## The endpoint excess $9/256$

Define $Y=1-\rho^2/\lambda^2\in[0,1]$. A Burkholder-Davis-Gundy estimate gives $\mathbb EY^3=O(\lambda^3)$, so
$\mathbb E\rho/\lambda=1-\tfrac12\mathbb EY-\tfrac18\mathbb EY^2+O(\lambda^3)$ with $\mathbb EY=1-m_2/\lambda^2$ and
$\mathbb EY^2=1-2m_2/\lambda^2+m_4/\lambda^4$:

In [6]:
lam = sp.symbols('lambda', positive=True)
m2 = 2*(lam-1+sp.exp(-lam))
m4 = sp.Rational(4,27)*(45*lam**2-156*lam+214-54*(lam+4)*sp.exp(-lam)+2*sp.exp(-3*lam))
EY = 1 - m2/lam**2; EY2 = 1 - 2*m2/lam**2 + m4/lam**4
Erho_over_lam = sp.series(1 - EY/2 - EY2/8, lam, 0, 3).removeO()
print('E rho / lambda =', Erho_over_lam)
EZ = sp.series(Erho_over_lam*lam/2, lam, 0, 3).removeO()          # E|Z_lambda| = E rho / 2
excess = sp.expand((lam/2 - sp.Rational(37,768)*lam**2) - EZ)
print('F - E|R.e|/p   =', excess, '   (9/256 =', sp.Rational(9,256), ')')

E rho / lambda = lambda**2/45 - lambda/6 + 1
F - E|R.e|/p   = 9*lambda**2/256    (9/256 = 9/256 )


In [7]:
ex = pd.read_csv(ROOT/'data'/'endpoint_excursion_diagnostic.csv')
ex = ex[ex['lambda'] <= 0.1].sort_values(['lambda','steps']).groupby('lambda').tail(1)
print(ex[['lambda','steps','n','excursion_over_lambda2','excursion_se']].to_string(index=False))
print('\nall within 2 SE of 9/256:', bool(np.all(np.abs(ex.excursion_over_lambda2 - 9/256) <= 2*ex.excursion_se)))

 lambda  steps      n  excursion_over_lambda2  excursion_se
  0.005    512 160000              0.03494475    0.00037573
  0.010    512 160000              0.03519042    0.00031264
  0.020    512  80000              0.03487143    0.00036764
  0.050    512  80000              0.03492859    0.00028682
  0.100   2048  60000              0.03470485    0.00026917

all within 2 SE of 9/256: True


## Generalization and dimension independence

The same argument applies to any stationary, sign-symmetric velocity diffusion with stationary density
$\varpi$ and local variance rate $a(0)$ at zero velocity:
$\mathbb E[\text{range}]=\mu t-\tfrac{37}{384}\varpi(0)a(0)t^2+o(t^2)$, $\mu=\mathbb E|U|$. For isotropic
rotational diffusion in $D$ dimensions (first tangent harmonic decaying as $e^{-t}$) one has
$\varpi_D(u)=c_D(1-u^2)^{(D-3)/2}$ and $a_D(0)=2/(D-1)$, and the identity $\varpi_D(0)a_D(0)=\mu_D$ makes the
*relative* coefficient dimension independent: $F_D=\mu_D\lambda[1-\tfrac{37}{384}\lambda+o(\lambda)]$.

In [8]:
D = sp.symbols('D', positive=True)
cD = sp.gamma(D/2)/(sp.sqrt(sp.pi)*sp.gamma((D-1)/2))
muD = sp.gamma(D/2)/(sp.sqrt(sp.pi)*sp.gamma((D+1)/2))
diff = sp.gammasimp(sp.combsimp((cD*2/(D-1))/muD))
print('ratio varpi_D(0) a_D(0) / mu_D =', diff, '  (symbolically, using Gamma((D+1)/2) = ((D-1)/2) Gamma((D-1)/2))')
print('numerically for D=2..6:', [float((cD*2/(D-1)-muD).subs(D, k)) for k in range(2, 7)])
print('mu_2 =', sp.simplify(muD.subs(D, 2)), '  mu_3 =', sp.simplify(muD.subs(D, 3)))

ratio varpi_D(0) a_D(0) / mu_D = 1   (symbolically, using Gamma((D+1)/2) = ((D-1)/2) Gamma((D-1)/2))
numerically for D=2..6: [0.0, 0.0, 0.0, 0.0, 0.0]
mu_2 = 2/pi   mu_3 = 1/2
